In [10]:
import json
import os
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from PIL import Image
import cv2
# ---------------- Настройки шрифта ----------------
TTF_FONT_PATH = "NotoSans.ttf"  # Путь к TTF с кириллицей
PDF_FONT_NAME = "MyFont"
pdfmetrics.registerFont(TTFont(PDF_FONT_NAME, TTF_FONT_PATH))

# ---------------- Функция конвертации ----------------
def structured_json_to_pdf(doc_json, output_path: str, font_size=12, image_dir=None):
    """
    doc_json - JSON документа
    image_dir - если есть картинки для блоков типа "Picture"
    """
    width = doc_json.get("width", 800)
    height = doc_json.get("height", 1000)
    c = canvas.Canvas(output_path, pagesize=(width, height))
    c.setFont(PDF_FONT_NAME, font_size)

    for block in doc_json["blocks"]:
        block_type = block.get("type", "Text")
        struct = block.get("structure")

        # ---------------- Picture ----------------
        if block_type.lower() == "image":
            # image filename = id или polygon можно сделать
            img_path = 'tmp.png'
            cv2.imwrite(img_path, struct['image_array'])
            if os.path.exists(img_path):
                # Получаем bounding box для вставки
                bbox = block.get("bbox")
                if bbox:
                    x1, y1, x2, y2 = bbox
                    img_height = y2 - y1
                    img_width = x2 - x1
                    # Инвертируем y
                    y_pdf = height - y2
                    c.drawImage(img_path, x1, y_pdf, width=img_width, height=img_height)
            continue

        # ---------------- Структуры ----------------
        if struct is None:
            continue

        struct_type = struct.get("type", "predictions")

        # -------- Линии текста --------
        if struct_type == "lines":
            for line in struct.get("lines", []):
                for pred in line.get("predictions", []):
                    x_min = min(p[0] for p in pred["polygon"])
                    y_min = min(p[1] for p in pred["polygon"])
                    y_pdf = height - y_min - font_size
                    c.drawString(x_min, y_pdf, pred["text"])

        # -------- Таблицы --------
        elif struct_type == "table":
            for row in struct.get("rows", []):
                for cell in row.get("cells", []):
                    poly = cell.get("polygon")
                    if poly:
                        x_min = min(p[0] for p in poly)
                        y_min = min(p[1] for p in poly)
                        y_pdf = height - y_min - font_size
                        text = cell.get("text", "")
                        c.drawString(x_min, y_pdf, text)

                        # Нарисовать рамку ячейки
                        x1 = min(p[0] for p in poly)
                        y1 = min(p[1] for p in poly)
                        x2 = max(p[0] for p in poly)
                        y2 = max(p[1] for p in poly)
                        c.rect(x1, height - y2, x2 - x1, y2 - y1, stroke=1, fill=0)

        # -------- Простые предсказания --------
        elif struct_type in ("predictions", "pred"):
            for pred in struct.get("predictions", []):
                x_min = min(p[0] for p in pred["polygon"])
                y_min = min(p[1] for p in pred["polygon"])
                y_pdf = height - y_min - font_size
                c.drawString(x_min, y_pdf, pred.get("text", ""))

    c.showPage()
    c.save()
    print(f"PDF сохранён: {output_path}")

# ---------------- Пример использования ----------------
if __name__ == "__main__":
    with open("1.json", "r", encoding="utf-8") as f:
        doc_json = json.load(f)

    structured_json_to_pdf(doc_json, "output.pdf", font_size=12, image_dir="images")

PDF сохранён: output.pdf
